# Семинар "Введение в LLM: OpenAI SDK + OpenRouter"

Мы пройдем путь от первого запроса к открытой бесплатной LLM до reasoning, streaming, structured output, tool calling, работы с изображениями и embeddings.

> Бесплатные модели, их лимиты и поддерживаемые возможности могут меняться. Если конкретный model ID временно недоступен, OpenRouter вернёт ошибку API.

## Почему здесь OpenAI SDK

Многие LLM-сервисы запускают модели с помощью [vLLM](https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html) или [SGLang](https://docs.sglang.ai/basic_usage/openai_api.html). Эти серверы предоставляют API, совместимый с OpenAI: к ним можно обращаться через Python-клиент `openai`, указав адрес сервера в `base_url`. На семинаре тот же подход используется для OpenRouter. Поддержка конкретных методов и параметров зависит от сервера и выбранной модели.

Какие модели существуют и что они умеют, можно посмотреть в [каталоге Hugging Face](https://huggingface.co/models): там есть языковые модели (LLM), модели для получения векторных представлений (embedding) и модели, работающие с изображениями и текстом (VLM).

[Artificial Analysis](https://artificialanalysis.ai/) помогает сравнивать модели по результатам тестов, скорости и стоимости использования через API. Его таблицы полезны для предварительного выбора модели, но итог зависит от конкретной задачи и провайдера: перед использованием стоит проверить модель на собственных примерах.

Для разных задач используют разные бенчмарки: MMLU-Pro проверяет знания и рассуждение по многим учебным дисциплинам, GPQA — ответы на сложные научные вопросы, LiveCodeBench — решение задач по программированию, а SWE-bench — исправление ошибок в реальных репозиториях. Для моделей, которые работают с изображениями, есть MMMU с вопросами по тексту и изображениям; для текстовых embedding-моделей — MTEB с задачами поиска, классификации и сравнения текстов. Поэтому при выборе модели смотрите на результаты тестов, близких к вашей задаче, а не только на общий рейтинг.

# 0. Доступ к моделям

На семинаре будем работать с бесплатными моделями с OpenRouter.

Для начала вам необходимо:
- зарегистрироваться на сайте https://openrouter.ai
- перейти во вкладку https://openrouter.ai/keys
- создать ключ через `+ New Key`
- скопировать созданный ключ и сохранить его у себя

## 1. Подключение к открытым бесплатным моделям через OpenRouter

OpenRouter предоставляет OpenAI-compatible API. Поэтому для всех примеров - chat, images и embeddings — используется один клиент из библиотеки `openai`, отличаются только model ID и содержимое запроса.

In [ ]:
!pip install vllm

In [30]:
import base64
import json
from getpass import getpass
from typing import Literal

import numpy as np
from openai import OpenAI
from pydantic import BaseModel, Field

In [60]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

api_key = getpass("OpenRouter API key: ").strip()
if not api_key:
    raise RuntimeError("Нужен личный ключ: https://openrouter.ai/keys")

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)

OpenRouter API key: ··········


## 2. Подача запроса в LLM

Главные параметры запроса:

- `model` — модель, которая обработает запрос;
- `messages` — история диалога с ролями `system`, `user`, `assistant` и `tool`;
- `temperature` — случайность ответа: малые значения делают результат стабильнее, большие — разнообразнее;
- `top_p` — альтернативный способ ограничить случайность. Обычно настраивают либо `temperature`, либо `top_p`, но не оба сразу;
- `max_tokens` — верхняя граница длины ответа;
- `stream` — возвращать ответ целиком или небольшими фрагментами;
- `response_format` — потребовать, например, результат по JSON Schema;
- `extra_body` — передать провайдеру дополнительные параметры, которых нет в стандартной сигнатуре OpenAI SDK.

In [63]:
# https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16
CHAT_MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"

In [70]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Отвечай кратко и по-русски."},
        {"role": "user", "content": "Объясни, что такое токен в LLM, одним абзацем."},
    ],
    temperature=0.2,
    max_tokens=200,
)
print(response)
choice = response.choices[0]
print(choice.message.content or "(пустой ответ)")
print("Usage:", response.usage)

ChatCompletion(id='gen-1791307437-kPm3meQoAeo7AA2vnUwp', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Токен\u202f—\u202fэто минимальная единица текста, которую модель LLM воспринимает и обрабатывает; это может быть слово, часть слова (субслово) или даже отдельный символ, в зависимости от используемой токенизационной схемы, и каждый токен преобразуется в числовой вектор (эмбеддинг), который проходит через нейронные слои модели для генерации или понимания контента.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="We need to respond in Russian, concise, one paragraph, explaining what a token is in LLM. Should be short, one paragraph. Let's craft.\n", reasoning_details=[{'type': 'reasoning.text', 'text': "We need to respond in Russian, concise, one paragraph, explaining what a token is in LLM. Should be short, one paragraph. Let's craft.\n", 'format': 'unknown', 'index': 0

In [7]:
print(json.dumps(response.model_dump(), ensure_ascii=False, indent=2))

{
  "id": "gen-1791210541-EACn21CgBj7RPPiRXPsn",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Токен в LLM — это минимальный фрагмент текста, на который модель разбивает вход для обработки: обычно это часть слова, целое слово, пробел или символ, и именно токены модель «видит», предсказывает и использует для подсчёта контекста и стоимости.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning": "We need answer user in Russian, brief one paragraph. Need explain token in LLM. Final one paragraph.\n",
        "reasoning_details": [
          {
            "type": "reasoning.text",
            "text": "We need answer user in Russian, brief one paragraph. Need explain token in LLM. Final one paragraph.\n",
            "format": "unknown",
            "index": 0
          }
   

## 3. Локальный запуск LLM через vLLM

Открытую модель можно запустить на своём компьютере или сервере и обращаться к ней через тот же OpenAI SDK. Для этого примера понадобится сервер с NVIDIA GPU и установленным vLLM.

vLLM и SGLang позволяют запустить модель как сервис: они предоставляют API и обрабатывают несколько запросов одновременно. Чтобы ускорить генерацию, такие серверы объединяют запросы в batch и эффективнее используют память GPU. Для отдельных экспериментов модель удобно запускать через `transformers`.

Для демонстрации используем модель [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B). Запустите её в терминале и дождитесь готовности сервера:

```bash
vllm serve Qwen/Qwen3-1.7B --reasoning-parser qwen3
```

Сервер будет доступен по адресу `http://localhost:8000/v1`. Если он запущен на другой машине, замените `localhost` на адрес этой машины.


In [5]:
LOCAL_MODEL = "Qwen/Qwen3-1.7B"
local_client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

local_response = local_client.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[
        {"role": "system", "content": "Отвечай кратко и по-русски."},
        {"role": "user", "content": "Объясни, что такое токен в LLM, одним абзацем."},
    ],
)
print(local_response.choices[0].message.content)



Токен в LLM — это единица текста, используемая для представления информации в модели, которая помогает ей понимать и генерировать текст. Каждый токен может быть словом или подсловом, и модель обрабатывает их в последовательности, чтобы выявлять синтаксис, семантику и контекст.


## 4. Запрос к локальной модели через curl

К OpenAI-совместимому API можно обращаться прямо из терминала. Отправим те же `messages` на endpoint `/v1/chat/completions`. Имя модели должно совпадать с моделью, которую запустили в vLLM. Также можно сходить и в модель с OpenRouter.

In [7]:
!curl http://localhost:8000/v1/chat/completions \
    -H 'Content-Type: application/json' \
    -d '{"model":"Qwen/Qwen3-1.7B","messages":[{"role":"system","content":"Отвечай кратко и по-русски."},{"role":"user","content":"Объясни, что такое токен в LLM, одним абзацем."}]}'

{"id":"chatcmpl-a58a671ed345be5f","object":"chat.completion","created":1791298151,"model":"Qwen/Qwen3-1.7B","choices":[{"index":0,"message":{"role":"assistant","content":"\n\nТокен в LLM — это единица текста, используемая для представления информации в модели, которая помогает ей понимать и генерировать текст. Каждый токен может быть словом или подсловом, и модель обрабатывает их в последовательности, чтобы выявлять синтаксис, семантику и контекст.","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":"\nOkay, the user wants a one-paragraph explanation of what a token is in an LLM. Let me start by recalling that in the context of large language models, a token is a unit of text. They are used in the model's processing, like when you input text, the model breaks it down into tokens. Each token is a word or subword, and the model uses these tokens to understand context. Also, tokens are part of the model's vocabulary, which is a large set of words it understand

## 5. Диалог с передачей контекста

Сервер не хранит историю разговора между запросами. Для каждого нового вопроса передаём список `messages` целиком: системную инструкцию, прошлые вопросы и ответы, затем текущий вопрос. После получения ответа добавляем его с ролью `assistant`, чтобы следующий запрос получил полный контекст. Длинная история сообщений может превысить лимит контекста модели — в таком случае старые сообщения нужно удалять или делать перефраз истории сообщений.

In [11]:
messages = [{"role": "system", "content": "Отвечай кратко и по-русски."}]

while True:
    question = input("Вопрос: ").strip()
    if question.lower() == "выход":
        break
    if not question:
        continue

    messages.append({"role": "user", "content": question})
    response = local_client.chat.completions.create(
        model=LOCAL_MODEL,
        messages=messages,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    answer = response.choices[0].message.content or ""
    messages.append({"role": "assistant", "content": answer})
    print(f"Модель: {answer}\n")

print('-' * 50)
print("Весь диалог:")
role_names = {"system": "Системный промпт", "user": "Пользователь", "assistant": "LLM"}
for message in messages:
    print(f"{role_names[message['role']]}: {message['content']}")


Вопрос: Что такое LLM?
Модель: LLM — это нейронная сеть, которая может понимать и генерировать текст.

Вопрос: Что такое нейронная сеть?
Модель: Нейронная сеть — это компьютерная модель, которая использует нейронные сети для обучения и принятия решений.

Вопрос: выход
--------------------------------------------------
Весь диалог:
Системный промпт: Отвечай кратко и по-русски.
Пользователь: Что такое LLM?
LLM: LLM — это нейронная сеть, которая может понимать и генерировать текст.
Пользователь: Что такое нейронная сеть?
LLM: Нейронная сеть — это компьютерная модель, которая использует нейронные сети для обучения и принятия решений.


## 6. Reasoning

Параметр передаётся через `extra_body`, потому что он не входит в стандартную сигнатуру Chat Completions в OpenAI SDK. Reasoning полезен там, где ответ требует нескольких связанных шагов: в математике, анализе кода, планировании, сравнении вариантов и проверке ограничений. Для простых вопросов он обычно только увеличивает задержку и расход токенов.

Способ включения зависит от модели и провайдера. Для Qwen в примере используется `chat_template_kwargs.enable_thinking`.

In [8]:
reasoning_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "У Пети вдвое больше яблок, чем у Маши. Вместе 18. Сколько у каждого?"}],
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True,
        },
    },
)
message = reasoning_response.choices[0].message
extra = message.model_extra or {}
reasoning = extra.get("reasoning")
if reasoning:
    print("REASONING:\n", reasoning)
else:
    print("Модель не вернула reasoning отдельным полем.")
print("\nANSWER:\n", message.content or "(пустой ответ)")

REASONING:
 We need answer in Russian likely. User: "У Пети вдвое больше яблок, чем у Маши. Вместе 18. Сколько у каждого?" Need solve simple algebra. Let Masha x, Petya 2x, total 3x=18, x=6, Petya 12. Need final concise.


ANSWER:
 У Маши **6** яблок, у Пети **12**.


## 7. Streaming

При `stream=True` ответ приходит небольшими фрагментами. Это уменьшает время до первого видимого текста, хотя полная генерация не обязательно становится быстрее. Streaming применяют в чатах, интерактивных ассистентах и генерации длинных текстов или кода, чтобы пользователь сразу видел прогресс. Получатель должен последовательно собрать фрагменты.

In [9]:
stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Назови три практических применения LLM."}],
    stream=True,
)
chunks = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chunks.append(chunk.choices[0].delta.content)
print(f"chunks: {chunks}")
print(f"Output {''.join(chunks)}")

chunks: ['Т', 'ри', ' прак', 'тических', ' применения', ' L', 'LM', ':', '\n\n', '1', '.', ' **', 'Ч', 'ат', '-', 'бот', 'ы', ' и', ' вир', 'ту', 'альные', ' асс', 'ист', 'енты', '**', ' —', ' ответы', ' на', ' вопросы', ' клиентов', ',', ' поддержка', ',', ' консультации', '.', '\n', '2', '.', ' **', 'А', 'в', 'том', 'ат', 'из', 'ация', ' тек', 'стов', 'ой', ' работы', '**', ' —', ' на', 'писание', ' emails', ',', ' резюме', ',', ' от', 'чё', 'тов', ',', ' статей', ' и', ' документов', '.', '\n', '3', '.', ' **', 'А', 'на', 'лиз', ' и', ' струк', 'тури', 'рование', ' данных', '**', ' —', ' из', 'влечение', ' информации', ' из', ' документов', ',', ' класс', 'иф', 'ика', 'ция', ' отзывов', ',', ' крат', 'кие', ' резюме', '.']
Output Три практических применения LLM:

1. **Чат-боты и виртуальные ассистенты** — ответы на вопросы клиентов, поддержка, консультации.
2. **Автоматизация текстовой работы** — написание emails, резюме, отчётов, статей и документов.
3. **Анализ и структурирование 

## 8. Structured output

Pydantic-модель задаёт поля ответа, их типы и ограничения. `chat.completions.parse()` преобразует её в строгую JSON Schema, передаёт серверу и разбирает ответ обратно в объект Pydantic. Нужно обрабатывать отказ модели, обрыв по лимиту и ошибку API. Возможность должна поддерживаться выбранной моделью и её текущим endpoint.

Structured output нужен, когда результат читает программа: для извлечения данных из документов, классификации, заполнения карточек, построения аргументов API и этапов пайплайна. Он уменьшает количество хрупкого парсинга, но не гарантирует фактическую правильность значений.

In [10]:
class LessonSummary(BaseModel):
    topic: str
    key_points: list[str] = Field(min_length=3, max_length=3)
    difficulty: Literal["beginner", "intermediate", "advanced"]


structured_response = client.chat.completions.parse(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Составь план пары по llm."}],
    response_format=LessonSummary,
)
message = structured_response.choices[0].message
lesson = message.parsed
print(lesson.model_dump_json(indent=2))

{
  "topic": "Основы больших языковых моделей (LLM)",
  "key_points": [
    "Краткая история и архитектурные принципы LLM, включая трансформеры",
    "Проблемы, на которые решают LLM: генерация текста, классификация, извлечение и диалоги",
    "Типовые этапы подготовки и использования модели: обучение, fine-tuning, промптирование и оценка"
  ],
  "difficulty": "intermediate"
}


## 9. Tool calling: модель предлагает, Python выполняет

Сначала модель возвращает `tool_calls`. Наш код проверяет имя и аргументы, выполняет разрешённую функцию и добавляет результат с ролью `tool`. Затем модель формулирует ответ человеку. Никогда не исполняйте произвольный код или неизвестное имя инструмента из ответа модели.

In [11]:
WEATHER = {
    "москва": {"temperature_c": 7, "condition": "облачно"},
    "казань": {"temperature_c": 5, "condition": "дождь"},
    "сочи": {"temperature_c": 18, "condition": "солнечно"},
}

def get_weather(city: str) -> dict:
    if not isinstance(city, str) or not city.strip():
        raise ValueError("city должна быть непустой строкой")
    normalized = city.strip().lower()
    return {
        "city": city,
        **WEATHER.get(normalized, {"error": "нет данных для этого города"}),
    }

In [12]:
TOOLS = {"get_weather" : get_weather}

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Получить погоду из учебного справочника",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Название города по-русски"},
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    },
}]

In [13]:
messages = [{"role": "user", "content": "Какая учебная погода сейчас в Казани?"}]

first = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)
assistant_message = first.choices[0].message
print("tool_calls:", assistant_message.tool_calls)

if not assistant_message.tool_calls:
    print(assistant_message.content or "Модель не запросила инструмент")
else:
    messages.append(assistant_message.model_dump())
    for tool_call in assistant_message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        result = TOOLS[tool_call.function.name](**arguments)
        print(f"Python выполнил функцию {tool_call.function.name} с аргументами {arguments}: {result}")
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result, ensure_ascii=False),
        })
    final = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        tools=tools,
    )
    print(final.choices[0].message.content or "(пустой ответ)")

tool_calls: [ChatCompletionMessageFunctionToolCall(id='1deb3e11-5478-9a8a-9983-b396af350927_0', function=Function(arguments='{"city":"Казань"}', name='get_weather'), type='function', index=0)]
Python выполнил функцию get_weather с аргументами {'city': 'Казань'}: {'city': 'Казань', 'temperature_c': 5, 'condition': 'дождь'}
Сейчас в Казани: **+5 °C**, дождь. 🌧️


## 10. Передача изображения в VLM

Изображение можно передать по публичному URL или как base64 data URL. Здесь мы скачиваем небольшую фотографию в память, проверяем её и кодируем в base64 — так модель получает сами байты изображения, а не внешнюю ссылку.

In [13]:
!curl -L "https://upload.wikimedia.org/wikipedia/commons/8/8f/The-Transformer-model-architecture.png" -o "transformer-architecture.png"
# !wget "https://upload.wikimedia.org/wikipedia/commons/8/8f/The-Transformer-model-architecture.png" -O "transformer-architecture.png"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 48342  100 48342    0     0   360k      0 --:--:-- --:--:-- --:--:--  363k


In [14]:
with open("transformer-architecture.png", "rb") as image_file:
    image_base64 = base64.b64encode(image_file.read()).decode("ascii")
image_data_url = f"data:image/png;base64,{image_base64}"

In [36]:
vision_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Кратко опиши что изображено на картинке",
            },
            {
                "type": "image_url",
                "image_url": {"url": image_data_url},
            },
        ],
    }],
)
print(vision_response.choices[0].message.content or "(пустой ответ)")

NotFoundError: Error code: 404 - {'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: qwen/qwen3.8-27b', 'code': 404}, 'user_id': 'user_3K5nnPmiaiOMN7VQHSL9zQPnGqO'}

### Ограничение локальной Qwen3: только текст

`Qwen/Qwen3-1.7B` — текстовая модель: у неё нет компонента для обработки изображений. OpenAI-совместимый формат запроса позволяет добавить `image_url` в сообщение, но сервер vLLM отклонит такой запрос для этой модели. Проверим это на небольшом PNG. Для анализа изображений нужна мультимодальная модель, например модель семейства Qwen3-VL.


In [15]:
local_vision_response = local_client.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Кратко опиши что изображено на картинке?"},
            {"type": "image_url", "image_url": {"url": image_data_url}},
        ],
    }],
)
print(local_vision_response.choices[0].message.content or "(пустой ответ)")

BadRequestError: Error code: 400 - {'error': {'message': 'Qwen/Qwen3-1.7B is not a multimodal model', 'type': 'BadRequestError', 'param': None, 'code': 400}}

## 11. Работа с embedding-моделями через OpenAI-клиент

Embedding — вектор фиксированной длины. Близкие по смыслу тексты обычно получают близкие направления в векторном пространстве. Для сравнения используем cosine similarity: чем значение ближе к `1`, тем направления похожее.

In [17]:
# https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16
TEXT_EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"

In [18]:
def cosine_similarity(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    return float(
        np.dot(vector_a, vector_b)
        / (np.linalg.norm(vector_a) * np.linalg.norm(vector_b))
    )


texts = [
    "what is the capital of France",
    "Paris is the capital and largest city of France",
    "Борщ варят из свёклы, капусты, картофеля и мяса, подают со сметаной",
]

embedding_response = client.embeddings.create(
    model=TEXT_EMBEDDING_MODEL,
    input=texts,
)
text_vectors = np.asarray([item.embedding for item in embedding_response.data])

for index in range(1, len(texts)):
    similarity = cosine_similarity(text_vectors[0], text_vectors[index])
    print(f"{texts[0]!r} | {texts[index]!r}: {similarity:.3f}")

'what is the capital of France' | 'Paris is the capital and largest city of France': 0.762
'what is the capital of France' | 'Борщ варят из свёклы, капусты, картофеля и мяса, подают со сметаной': 0.111


## 12. VL embeddings: текст, изображение и их сочетание

VL embedding-модель принимает текст, изображение или оба вида данных в одном `content`. В последнем случае она создаёт **один эмбеддинг для пары «текст + изображение»**. Все эти векторы можно сравнивать через cosine similarity: чем выше значение, тем ближе представления в пространстве модели. Это мера близости, а не вероятность.

Сравним схему Transformer из предыдущего раздела со [схемой CNN](https://commons.wikimedia.org/wiki/File:Convolutional_Neural_Network.png). Сначала получим эмбеддинги для всех вариантов одним запросом, затем рассмотрим три сравнения.


In [20]:
# https://huggingface.co/nvidia/llama-nemotron-embed-vl-1b-v2
VL_EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

In [30]:
!curl -fL "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg" -o "cat-photo.jpg"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  273k  100  273k    0     0  1050k      0 --:--:-- --:--:-- --:--:-- 1054k


In [31]:
with open("cat-photo.jpg", "rb") as image_file:
    cat_base64 = base64.b64encode(image_file.read()).decode("ascii")
cat_data_url = f"data:image/jpeg;base64,{cat_base64}"

vl_inputs = [
    {
        "content": [
            {"type": "text", "text": "Transformer architecture diagram with encoder and decoder"},
        ],
    },
    {
        "content": [
            {"type": "image_url", "image_url": {"url": image_data_url}},
        ],
    },
    {
        "content": [
            {"type": "image_url", "image_url": {"url": cat_data_url}},
        ],
    },
    {
        "content": [
            {"type": "text", "text": "Transformer architecture"},
            {"type": "image_url", "image_url": {"url": image_data_url}},
        ],
    },
    {
        "content": [
            {"type": "text", "text": "A photo of an orange cat"},
            {"type": "image_url", "image_url": {"url": cat_data_url}},
        ],
    },
]
vl_embedding_response = client.embeddings.create(
    model=VL_EMBEDDING_MODEL,
    input=vl_inputs,
)

(
    text_embed,
    transformer_img_embed,
    cat_img_embed,
    transformer_text_img_embed,
    cat_text_img_embed,
) = [
    np.asarray(item.embedding)
    for item in sorted(vl_embedding_response.data, key=lambda item: item.index)
]

### Пример 1. Текст и изображения

Один и тот же текст сравниваем с двумя изображениями. Ожидаем, что описание Transformer будет ближе к схеме Transformer, но конкретные значения зависят от модели.


In [32]:
transformer_score = cosine_similarity(text_embed, transformer_img_embed)
cat_score = cosine_similarity(text_embed, cat_img_embed)
print(f"Текст про Transformer | схема Transformer: {transformer_score:.3f}")
print(f"Текст про Transformer | фото кота: {cat_score:.3f}")

Текст про Transformer | схема Transformer: 0.269
Текст про Transformer | фото кота: 0.150


### Пример 2. Пара «текст + изображение» и другая такая пара

В каждом элементе `content` находятся и подпись, и изображение. Модель превращает их в один общий вектор, после чего сравниваем две пары.


In [33]:
mixed_score = cosine_similarity(
    transformer_text_img_embed,
    cat_text_img_embed,
)
print(f"Текст + схема Transformer | текст + фото кота: {mixed_score:.3f}")

Текст + схема Transformer | текст + фото кота: 0.142


### Пример 3. Изображение и изображение

Сравниваем эмбеддинги двух схем без текстовых подписей.

In [34]:
image_score = cosine_similarity(transformer_img_embed, cat_img_embed)
print(f"Схема Transformer | фото кота: {image_score:.3f}")

Схема Transformer | фото кота: 0.148


## Итог

- OpenRouter позволяет обращаться к разным открытым моделям через единый OpenAI-compatible клиент.
- Возможности определяет выбранная модель: проверяйте поддержку reasoning, tools, structured output и изображений.
- Streaming ускоряет появление первого текста, но не обязательно всю генерацию.
- Tool calling не исполняет функцию: модель предлагает вызов, приложение проверяет и выполняет его.
- Text и VL embeddings позволяют сравнивать смысл текстов и изображений в векторном пространстве.
- Бесплатные модели и лимиты меняются, поэтому обрабатывайте ошибки и проверяйте актуальные model ID.


## Полезные ссылки

- [OpenAI Python SDK на GitHub](https://github.com/openai/openai-python) — примеры использования клиента и справочник по его возможностям.
- [Каталог моделей Hugging Face](https://huggingface.co/models) — поиск LLM, embedding-моделей и VLM по задачам.
- [Документация vLLM](https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html) — запуск OpenAI-совместимого сервера и параметры API.
- [Документация SGLang](https://docs.sglang.ai/basic_usage/openai_api.html) — запуск сервера и OpenAI-совместимые запросы.
